# Encontro 6 · Da base suja à base pronta para o modelo

**Coleta e Preparação de Dados · PUCRS Online** — prática da Aula 6 (limpeza, integração, redução e transformação)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucaskup/pucrs-coleta-e-preparacao/blob/main/aula6/pratica/encontro6.ipynb)

A Aurora Imóveis fechou parceria com outra imobiliária fictícia, a **Casa Sul**, e quer uma base única
para treinar um modelo de preço. Vocês vão montar o pipeline em **estações**: cada grupo é dono de
uma etapa, registra as decisões em um **diário** e, na apresentação, outro grupo audita esse diário.

| Fonte | Arquivo | O que tem |
| --- | --- | --- |
| Aurora | `aula6/imoveis_aurora.csv` | a base do Encontro 4, com todos os problemas |
| Casa Sul | `aula6/casa_sul.json` | 80 anúncios, outro esquema, JSON aninhado |
| Referência | `aula6/bairros_poa.csv` | bairro → zona → cidade (zona é um agrupamento didático) |

| Estação | Tema | Começa de |
| --- | --- | --- |
| 1 | Limpeza: ausentes, sentinelas, inválidos, inconsistências, duplicatas | `aurora` bruto |
| 2 | Outliers: detectar, investigar e decidir | `aurora` bruto |
| 3 | Integração: esquema, conflitos de valor e registros duplicados entre fontes | `aurora` e `casa_sul` brutos |
| 4 | Transformação e redução: divisão treino/teste, normalização, discretização, PCA | `base_pronta()` |

Todos os grupos fazem também a **Parte final: o pipeline do colega** (10 min).

In [ ]:
import json
import re
import unicodedata
from urllib.request import urlopen

import numpy as np
import pandas as pd

BASE = "https://lucaskup.github.io/pucrs-coleta-e-preparacao/aula6"
aurora = pd.read_csv(f"{BASE}/imoveis_aurora.csv")
with urlopen(f"{BASE}/casa_sul.json") as resp:
    bruto_cs = json.load(resp)
bairros = pd.read_csv(f"{BASE}/bairros_poa.csv")
print(aurora.shape, len(bruto_cs["imoveis"]), bairros.shape)
print(bruto_cs["dicionario"])

def base_pronta():
    """Versão já limpa e integrada da base Aurora (ponto de partida da Estação 4)."""
    d = aurora.drop_duplicates().copy()
    d["conservacao"] = d["conservacao"].str.lower().replace({"otimo": "ótimo"})
    d["bairro"] = d["bairro"].replace({"Petropolis": "Petrópolis"})
    d["ano_construcao"] = d["ano_construcao"].replace(0, np.nan)
    d.loc[d["quartos"] < 0, "quartos"] = np.nan
    d.loc[d["codigo"] == "AUR-0060", "preco"] /= 10
    d.loc[d["codigo"] == "AUR-0199", "area_m2"] /= 10
    d.loc[d["codigo"] == "AUR-0147", "area_m2"] = np.nan
    return d.dropna(subset=["preco"]).reset_index(drop=True)

---
## Estação 1 — Limpeza

1. Quantas linhas duplicadas? Remova-as.
2. Encontre **todas** as codificações de ausência (vazios, sentinelas) e de inconsistência (grafias).
3. Encontre valores **inválidos** (fora do domínio possível).
4. Condomínio: as casas têm `0` e alguns apartamentos estão vazios. São a mesma coisa? O vazio é
   MCAR, MAR ou MNAR? (Compare o ano de construção de quem tem e de quem não tem o valor.)
5. Proponha a imputação do condomínio **e** crie um indicador de ausência.

In [ ]:
limpa = aurora.copy()
...

#### Diário de decisões — Estação 1
| Decisão | Justificativa | Alternativa descartada | Impacto |
| --- | --- | --- | --- |
| | | | |

---
## Estação 2 — Outliers

1. Calcule `preco_m2` e aplique **três** critérios em `preco`, `area_m2` e `preco_m2`: regra de 1,5·IQR,
   \|z\| > 3 e z robusto (mediana e MAD, com \|z\| > 3,5). Quantos candidatos cada um aponta?
2. Investigue os casos extremos: são **erros** (o que provavelmente aconteceu?) ou **valores legítimos raros**?
3. Decida, para cada caso: remover, corrigir, limitar (*capping*) ou manter e sinalizar. Justifique.

In [ ]:
d = aurora.drop_duplicates().copy()
d["preco_m2"] = d["preco"] / d["area_m2"]
...

#### Diário de decisões — Estação 2
| Registro | Critério que apontou | Erro ou legítimo? | Decisão | Justificativa |
| --- | --- | --- | --- | --- |
| | | | | |

---
## Estação 3 — Integração

1. Normalize o JSON da Casa Sul (`pd.json_normalize(bruto_cs["imoveis"])`) e leia o `dicionario`.
2. Monte o **mapeamento de esquema** para as colunas da Aurora: `preco`, `area_m2`, `quartos`, `vagas`,
   `condominio`, `conservacao`, `aceita_pet`, `data_anuncio`, `bairro`, `endereco`, `comodidades`.
3. Resolva os **conflitos de valor**: unidade do preço, metragem como texto, códigos de conservação,
   `S/N`, formato da data, nomes de bairro abreviados, comodidades em maiúsculas e sem acento.
   Há **um** anúncio cujo valor claramente não está em milhares. Qual?
4. **Registros duplicados entre fontes**: o mesmo imóvel pode estar anunciado nas duas imobiliárias,
   com preços um pouco diferentes e endereço escrito de outro jeito. Use *blocking* por
   (`bairro`, `quartos`) e compare endereço (Jaccard de bigramas) e área. Quantos pares você encontra?

In [ ]:
cs = pd.json_normalize(bruto_cs["imoveis"])
cs.head()

In [ ]:
...

In [ ]:
# registros duplicados entre as fontes
...

#### Diário de decisões — Estação 3
| Conflito | Regra adotada | Como verificamos |
| --- | --- | --- |
| | | |

---
## Estação 4 — Transformação e redução

Parta de `base_pronta()`. O alvo de um futuro modelo é o `preco`.

1. Separe **treino e teste** (80/20, `random_state=42`) **antes** de qualquer transformação.
2. Padronize (z-score) os atributos numéricos usando média e desvio **do treino**; aplique no teste.
3. Discretize o preço em 4 faixas de **mesma frequência** usando os cortes do treino; rotule
   (`econômico`, `médio`, `alto`, `alto padrão`). O teste ficou com 25% em cada faixa? Por quê?
4. Generalize `bairro` para `zona` com `bairros_poa.csv` (hierarquia de conceitos).
5. Rode uma **PCA** nos atributos padronizados (sem o preço). Quantos componentes explicam 80% da variância?

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

dados = base_pronta()
numericos = ["area_m2", "quartos", "banheiros", "vagas", "condominio", "iptu_anual", "ano_construcao"]
...

---
## Parte final (todos os grupos, 10 min) — O pipeline do colega

O código abaixo produz uma avaliação **otimista demais**. Encontre os pontos em que informação do
teste (ou do próprio alvo) vaza para o treino.

In [ ]:
# ======== código do colega: NÃO edite esta célula ========
from sklearn.model_selection import train_test_split
dados = base_pronta()
X = dados[["area_m2", "quartos", "banheiros", "vagas", "condominio", "iptu_anual"]]
X = X.fillna(X.mean())                                         # 1
X = (X - X.mean()) / X.std()                                   # 2
X["preco_medio_bairro"] = dados.groupby("bairro")["preco"].transform("mean")   # 3
X["faixa"] = pd.qcut(dados["preco"], 4, labels=False)          # 4
X_tr, X_te, y_tr, y_te = train_test_split(X, dados["preco"], test_size=0.2, random_state=42)
X_tr.head()

**Vazamentos encontrados:** …